# Notebook 4 — Model Implementation & Comparison
## Step 4: Train 5 models, tune, evaluate, compare

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import json
from pathlib import Path

from models import (
    train_logistic_regression, train_random_forest, train_xgboost,
    train_isolation_forest, train_autoencoder, autoencoder_predict,
    evaluate, comparison_table, plot_confusion_matrix, plot_roc_curves,
    save_training_configs
)

# Load preprocessed data
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'Class balance (test): normal={( y_test==0).sum()}  attack={(y_test==1).sum()}')

## 4.1 Baseline — Logistic Regression

In [ ]:
lr = train_logistic_regression(X_train, y_train, save_dir='../models')
lr_results = evaluate(lr, X_test, y_test, 'Logistic Regression')
plot_confusion_matrix(lr, X_test, y_test, 'Logistic Regression', save_dir='../reports')
lr_results

## 4.2 Random Forest

In [ ]:
# tune=False for quick run; set True for full GridSearchCV
rf = train_random_forest(X_train, y_train, tune=False, save_dir='../models')
rf_results = evaluate(rf, X_test, y_test, 'Random Forest')
plot_confusion_matrix(rf, X_test, y_test, 'Random Forest', save_dir='../reports')
rf_results

## 4.3 XGBoost (Best Model)

In [ ]:
xgb_model = train_xgboost(X_train, y_train, tune=False, save_dir='../models')
xgb_results = evaluate(xgb_model, X_test, y_test, 'XGBoost')
plot_confusion_matrix(xgb_model, X_test, y_test, 'XGBoost', save_dir='../reports')
xgb_results

## 4.4 Isolation Forest (Unsupervised)

In [ ]:
# Estimate contamination from training label ratio
contamination = float((y_train == 1).sum() / len(y_train))
print(f'Contamination: {contamination:.3f}')

iso = train_isolation_forest(X_train, contamination=contamination, save_dir='../models')
iso_results = evaluate(iso, X_test, y_test, 'Isolation Forest')
plot_confusion_matrix(iso, X_test, y_test, 'Isolation Forest', save_dir='../reports')
iso_results

## 4.5 Autoencoder (Deep Learning)

In [ ]:
# Train autoencoder only on normal traffic (index 0)
X_train_normal = X_train[y_train == 0]
X_val_normal   = X_train_normal[:500]  # quick validation split

autoencoder, ae_threshold = train_autoencoder(
    X_train_normal, X_val_normal,
    encoding_dim=16, epochs=30, batch_size=256,
    save_dir='../models'
)

ae_preds, ae_errors = autoencoder_predict(autoencoder, X_test, ae_threshold)

# Manual evaluation (autoencoder has no predict_proba)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
ae_results = {
    'model':     'Autoencoder',
    'accuracy':  round(accuracy_score(y_test, ae_preds), 4),
    'f1':        round(f1_score(y_test, ae_preds, zero_division=0), 4),
    'auc_roc':   round(roc_auc_score(y_test, ae_errors /
                       (ae_errors.max() + 1e-9)), 4),
}
print(ae_results)

## 4.6 Model Comparison

In [ ]:
all_results = [lr_results, rf_results, xgb_results, iso_results, ae_results]
comparison_df = comparison_table(all_results)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

comparison_df

In [ ]:
# Grouped bar chart
metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
melted = comparison_df[metrics].reset_index().melt(id_vars='model', var_name='metric', value_name='score')

plt.figure(figsize=(12, 5))
sns.barplot(data=melted, x='metric', y='score', hue='model', palette='Set2')
plt.ylim(0.8, 1.01)
plt.title('Model Comparison — All Metrics')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150)
plt.show()

In [ ]:
# ROC curves
supervised_models = {
    'Logistic Regression': lr,
    'Random Forest':        rf,
    'XGBoost':              xgb_model,
    'Isolation Forest':     iso,
}
plot_roc_curves(supervised_models, X_test, y_test, save_dir='../reports')

from IPython.display import Image
Image('../reports/roc_curves.png')

In [ ]:
# Save training configs for reproducibility
configs = {
    'logistic_regression': {'max_iter': 1000, 'C': 1.0, 'solver': 'lbfgs'},
    'random_forest':        {'n_estimators': 200, 'random_state': 42},
    'xgboost':              {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1},
    'isolation_forest':     {'n_estimators': 200, 'contamination': float(contamination)},
    'autoencoder':          {'encoding_dim': 16, 'epochs': 30, 'batch_size': 256},
    'best_model':           'XGBoost',
    'random_seed':          42,
}
save_training_configs(configs, save_dir='../models')
print('Configs saved to ../models/configs.yaml')

# Save comparison table
comparison_df.to_csv('../reports/model_comparison.csv')
print('Comparison table saved.')